<h2>Impact (v0.64) - 'Impact (v0.64).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 'v0.61' made use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less. A new 'composite' weight field has been added in Trade (v0.61) which only introduces the VtW-derived values where 'netWgt' is observed to be zero, however this resulted in only a tiny increase in recorded weight of around 1% for Angola, whereas the full VtW-derived field was ~50% greater. 

'v0.62' makes use of a new 'transport cost' field derived using ad-valorem transport cost rates identified from UNCTAD's Trade-and-Transport database. With these, we can now introduce another results disaggregation that presents NZF costs relative to expected transport costs, which is then fully in-line with how impacts are expressed in UNCTAD's CIA Task 3 work. 'v0.63' includes an assessment of 'Cost Intensity' matching process (similar to CIF-FOB matching), plus metrics on 'fac_vessel_types' and 'average sea-distance' (introduced in Trade v0.63). 'v0.64' has been modified throughout to account for a new scenario, namely the 'Japan' proposal, plus to remove the 'Low-then-High' scenario. In terms of the Impact notebook, this remains the same as 'v0.63'.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

version = "v0.64"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
voys_fem_dir = "/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments (v0.2)/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_v0.2.csv"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"
res_mlc = False

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "430"
iso_country = "Liberia"
year = 2025
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 5.250*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."
os.makedirs(output_dir+"{0}/results/".format(iso_country)) if not os.path.exists(output_dir+"{0}/results/".format(iso_country)) else "Output results folder already exists."

## 1.2 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv(voys_fem_dir, dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,0,5034525,413781699,CNTAX,156,CHN,CN,China,Taixing,CNNKG,156,CHN,CN,China,Nanjing,2018-01-01 06:00:00,2018-01-01 19:00:00,13.0,1.0,14.0,62.568523,0.0,0.0,0.0,3.635661,0.338483,3.974144,0.0,0.0,0.0,0.0,0.0,0.0,0.155243,0.014453,11.655930,1.085176,12.741107,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,0.169696,Refrigerated bulk,3786.0,dwt,3881.0,NaN,NaN,NaN,NaN,2,2000-5999,1959.0,Panama,SSD,4,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,242828.439338,7661.320668,7934.549945,7805.304846,7691.950531,7580.696290,7471.542086,7360.199865,7255.307537,7148.350725,7667.458038,12670.391275,12727.642802,14093.647297,14570.027623,14398.931835,14189.590704,14139.488873,14161.462401,7665.715352,8412.516422,8946.6

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_{1}.csv".format(iso_country, version), dtype=trade_im_d)
else:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_mlc_{1}.csv".format(iso_country, version), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Liberia': 

Number of trade records present in dataset: 15882
Number of partner countries present in dataset: 147
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1887.9m
Total weight within the dataset: 2018.0 million kilograms

Distance to market: 253.5 nm (min) to 13541.5 nm (max), 6763.9 nm (observed mean)
Average distance to market (volume-weighted): 4504.8 nm
Average distance to market (value-weighted): 5949.8 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,430,Liberia,156,China,000,World,Import,Sea,690590 | Ceramic chimney-pots,690590,kg,5134.200,5134.20,2067.83487,NaN,2067.83487,China | 690590 | Ceramic chimney-pots,69,69 | Ceramic products,China | 69 | Ceramic products,6905,"6905 | Roofing tiles, chimn","China | 6905 | Roofing tiles, chimn",69,6905,690590,"Chimney-pots, cowls, chimney liners, architect...",Mining and Quarrying,2693,Manufacture of structural non-refractory clay ...,66242,"Roofing tiles, chimney-pots, cowls, chimney li...",34,NMM - Mineral products n.e.c.,3735,"Non-refractory ceramic building bricks, floori...",0.2779,3.5989,both,2.482887,7441.930914,5134.20,[Container],['Container'],20405.0,12679.075255,94356.802097
1,1,430,Liberia,324,Guinea,000,World,Import,Sea,690721 | Ceramic tiles; flags,690721,mÃÂ²,1.902,16458.64,5746.21145,NaN,5746.21145,Guinea | 690721 | Ceramic tiles; flags,69,69 | Ceramic products,Guinea | 69 | Ceramic products,6907,6907 | Ceramic flags and pa,Guinea | 6907 | Ceramic flags and pa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4382,2.2820,both,2.864259,13112.854529,16458.64,[all],['all'],894.0,555.505674,7284.265093


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 302
Total size of the fleet: 196 vessels
6 vessel types contained in the dataset: ['Refrigerated bulk' 'Bulk carrier' 'Container' 'General cargo'
 'Chemical tanker' 'Vehicle']
Total transport work capacity: 25586.2 million deadweight-tonne nautical miles
Total cargo transported: 6807401.6 tonnes
Total transport work: 14600.6 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 1731.4 EJ
Total CO2 produced: 133544.8 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,11898,7819759,616401000,GWOXB,624,GNB,GW,Guinea-Bissau,Bissau,LRMLW,430,LBR,LR,Liberia,Monrovia,2018-03-17 14:00:00,2018-04-05 07:00:00,449.0,265.0,714.0,626.555873,5.122149,0.003412,5.125561,144.6840,85.3290,230.0130,0.0,0.0,0.0,0.0,0.0,0.0,6.383917,3.643685,479.807276,273.575398,753.382674,002,Africa,202,Sub-Saharan Africa,low_income,1,1,002,Africa,202,Sub-Saharan Africa,low_income,1,0,10.027603,Refrigerated bulk,1753.0,dwt,2225.0,12.0,NaN,NaN,NaN,2,2000-5999,1979.0,Moldova,MSD,1,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,1.394087e+06,43983.917910,45552.537026,44810.536258,44159.765056,43521.050452,42894.392234,42255.172531,41652.981901,41038.938989,44019.152769,72741.172684,73069.855764,80912.136773,83647.053381,82664.786297,81462.951323

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_arr_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_arr_voys_vess = int_arr_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_arr_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

                if int_arr_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_arr_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the importer.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "direct35_usd_dwtnm_"+year))
    
    if cost_int_match_desc_bool == False:
        trade_im["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True
    
    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd
    trade_im["direct_usd_"+year] = direct_usd

print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print("\nNumber of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.direct_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_agg = trade_im.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg["vol_kg_vtw_pct"] = round(100 * trade_im_agg.vol_kg_vtw / trade_im_agg.vol_kg_vtw.sum(), 1)
trade_im_agg["cifvalue_pct"] = round(100 * trade_im_agg.cifvalue / trade_im_agg.cifvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_im_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5)
))
trade_im_agg_2 = trade_im.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg_2["vol_kg_vtw_pct"] = round(100 * trade_im_agg_2.vol_kg_vtw / trade_im_agg_2.vol_kg_vtw.sum(), 1)
trade_im_agg_2["cifvalue_pct"] = round(100 * trade_im_agg_2.cifvalue / trade_im_agg_2.cifvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_im_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Oil tanker' 'Liquefied gas tanker']

Global voyages table used for the following vessel types found: 

['Other liquids tankers' 'Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['6907 | Ceramic flags and pa' '6914 | Ceramic articles; n.'
 '1404 | Vegetable products n' '1509 | Olive oil and its fr'
 '8711 | Motorcycles (includi' '7019 | Glass fibres (includ'
 '1605 | Crustaceans, mollusc' '1704 | Sugar confectionery '
 '1806 | Chocolate and other ' '7229 | Wire of other alloy '
 '1905 | Bread, pastry, cakes' '7326 | Iron or steel; artic'
 '7419 | Copper; articles the' '7616 | Aluminium; articles '
 '8215 | Cutlery; spoons, f

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,cifvalue_pct
0,['Container'],1. Primary Int. Inv. with Vess Type,27.0,41.7
1,['Oil tanker'],2. Global Inv. with Vess Type,24.0,16.0
2,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,20.7,14.9
3,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,19.9,2.4
4,"['Container', 'Bulk carrier']",1. Primary Int. Inv. with Vess Type,2.1,2.2


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,430,Liberia,156,China,000,World,Import,Sea,690590 | Ceramic chimney-pots,690590,kg,5134.200,5134.20,2067.83487,NaN,2067.83487,China | 690590 | Ceramic chimney-pots,69,69 | Ceramic products,China | 69 | Ceramic products,6905,"6905 | Roofing tiles, chimn","China | 6905 | Roofing tiles, chimn",69,6905,690590,"Chimney-pots, cowls, chimney liners, architect...",Mining and Quarrying,2693,Manufacture of structural non-refractory clay ...,66242,"Roofing tiles, chimney-pots, cowls, chimney li...",34,NMM - Mineral products n.e.c.,3735,"Non-refractory ceramic building bricks, floori...",0.2779,3.5989,both,2.482887,7441.930914,5134.20,[Container],['Container'],20405.0,12679.075255,94356.802097,1. Primary Int. Inv. with Vess Type,367.127861,366.315376,366.399675,366.399675,366.775024,366.775024,385.049116,696.25209,416.416426,395.448734,396.695752,405.009155,376.569768,675.928500,443.097388,409.140784,398.835166,418.305378,369.133404,750.886622,553.679721,485.215925,447.418634,436.652047,361.832809,793.660196,650.712980,599.852370,477.660771,468.851325,354.669782,815.397850,789.364879,803.127549,509.343868,492.873141,347.363522,797.988693,748.178216,810.241774,630.898650,502.253350,340.482034,699.643985,738.510237,770.199911,551.608249,508.225747,333.465215,643.142722,650.191255,749.140510,631.656206,511.986339
1,1,430,Liberia,324,Guinea,000,World,Import,Sea,690721 | Ceramic tiles; flags,690721,mÃÂ²,1.902,16458.64,5746.21145,NaN,5746.21145,Guinea | 690721 | Ceramic tiles; flags,69,69 | Ceramic products,Guinea | 69 | Ceramic products,6907,6907 | Ceramic flags and pa,Guinea | 6907 | Ceramic flags and pa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.4382,2.2820,both,2.864259,13112.854529,16458.64,[all],['all'],894.0,555.505674,7284.265093,3. Primary Int. Inv. without Vess Type,25.084711,25.046515,25.050347,25.050347,25.067678,25.067899,26.020753,42.33907,27.561459,26.156396,26.221587,27.026690,25.576889,40.997396,28.855152,27.222329,26.753885,27.705551,25.187901,45.235680,34.589798,31.662539,29.753862,28.644809,24.806065,49.274597,39.885437,37.620434,31.231215,30.324299,24.431506,49.809583,47.251208,49.031483,33.761701,31.558664,24.049460,47.803689,45.893112,48.946067,39.514719,32.019927,23.689573,43.515154,45.722064,48.502554,37.843945,32.315818,23.322640,40.949130,41.644835,47.466593,39.864007,32.508489


## 2.4 Create Results Tables

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,33253167.53,32606678.67,32045517.34,31494584.57,30955232.71,30405994.60,29888892.88,29362528.42
1,Levy,56674907.68,55069792.69,60815554.38,65899439.25,66723253.15,64268240.08,57543175.42,53519844.89
2,'as-is',35497740.02,37406550.71,45632833.47,53211949.57,63958588.57,61371618.02,60088883.93,54330515.66
3,Brazil,33664938.08,34948280.75,41035414.29,49675627.30,65858483.74,65829778.31,64056932.54,62470739.05
4,LISCR,33800606.43,34243219.20,38241181.08,40408242.92,43500742.08,52119037.57,48095987.00,52468394.21
5,Direct,34708175.94,35676035.01,37044040.72,39472087.57,41272658.78,41952502.57,42383551.97,42858308.90



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.76,1.73,1.70,1.67,1.64,1.61,1.58,1.56
1,Levy,3.00,2.92,3.22,3.49,3.53,3.40,3.05,2.83
2,'as-is',1.88,1.98,2.42,2.82,3.39,3.25,3.18,2.88
3,Brazil,1.78,1.85,2.17,2.63,3.49,3.49,3.39,3.31
4,LISCR,1.79,1.81,2.03,2.14,2.30,2.76,2.55,2.78
5,Direct,1.84,1.89,1.96,2.09,2.19,2.22,2.25,2.27



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.633,0.621,0.610,0.600,0.590,0.579,0.569,0.559
1,Levy,1.080,1.049,1.158,1.255,1.271,1.224,1.096,1.019
2,'as-is',0.676,0.713,0.869,1.014,1.218,1.169,1.145,1.035
3,Brazil,0.641,0.666,0.782,0.946,1.254,1.254,1.220,1.190
4,LISCR,0.644,0.652,0.728,0.770,0.829,0.993,0.916,0.999
5,Direct,0.661,0.680,0.706,0.752,0.786,0.799,0.807,0.816


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)
    
print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,23421740.15,22463114.02,28770037.04,34404854.67,35768020.45,33862245.48,27654282.53,24157316.47
1,'as-is',2244572.48,4799872.04,13587316.13,21717365.00,33003355.86,30965623.42,30199991.05,24967987.24
2,Brazil,411770.55,2341602.09,8989896.96,18181042.72,34903251.03,35423783.71,34168039.65,33108210.63
3,LISCR,547438.90,1636540.53,6195663.74,8913658.34,12545509.38,21713042.97,18207094.11,23105865.79
4,Direct,1455008.41,3069356.34,4998523.39,7977503.00,10317426.08,11546507.97,12494659.09,13495780.48



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,70.43,68.89,89.78,109.24,115.55,111.37,92.52,82.27
1,'as-is',4.38,9.41,15.60,25.33,33.33,37.97,41.80,45.96
2,Brazil,6.75,14.72,42.40,68.96,106.62,101.84,101.04,85.03
3,LISCR,1.24,7.18,28.05,57.73,112.75,116.50,114.32,112.76
4,Direct,1.65,5.02,19.33,28.30,40.53,71.41,60.92,78.69



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.24,1.19,1.52,1.82,1.89,1.79,1.46,1.28
1,'as-is',0.08,0.16,0.26,0.42,0.55,0.61,0.66,0.71
2,Brazil,0.12,0.25,0.72,1.15,1.75,1.64,1.60,1.32
3,LISCR,0.02,0.12,0.48,0.96,1.85,1.88,1.81,1.75
4,Direct,0.03,0.09,0.33,0.47,0.66,1.15,0.96,1.22



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.446,0.428,0.548,0.655,0.681,0.645,0.527,0.460
1,'as-is',0.028,0.058,0.095,0.152,0.197,0.220,0.238,0.257
2,Brazil,0.043,0.091,0.259,0.414,0.629,0.590,0.575,0.476
3,LISCR,0.008,0.045,0.171,0.346,0.665,0.675,0.651,0.631
4,Direct,0.010,0.031,0.118,0.170,0.239,0.414,0.347,0.440


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
else:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_mlc_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Liberia': 

Number of trade records present in dataset: 368
Number of partner countries present in dataset: 58
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$182.7m
Total weight within the dataset: 258.5 million kilograms

Distance to market: 253.5 nm (min) to 13542.2 nm (max), 4515.4 nm (observed mean)
Average distance to market (volume-weighted): 4053.5 nm
Average distance to market (value-weighted): 4826.2 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,430,Liberia,694,Sierra Leone,000,World,Export,Sea,691110 | Tableware and kitche,691110,kg,5.0900,5.0900,NaN,1658.831,1658.831,Sierra Leone | 691110 | Tableware and kitche,69,69 | Ceramic products,Sierra Leone | 69 | Ceramic products,6911,"6911 | Tableware, kitchenwa","Sierra Leone | 6911 | Tableware, kitchenwa",69,6911,691110,"Tableware & kitchenware, of porcelain/china",Mining and Quarrying,2691,Manufacture of non-structural non-refractory c...,66611,Tableware & kitchenware of porcelain/china,34,NMM - Mineral products n.e.c.,37221,"Ceramic tableware, kitchenware, other househol...",4.3093,0.2321,both,0.003068,385.014675,5.0900,[Container],['Container'],677.0,420.668167,161.963418
1,1,430,Liberia,842,USA,000,World,Export,Sea,691190 | Household and toilet,691190,kg,2059.8212,2059.8212,NaN,12216.000,12216.000,USA | 691190 | Household and toilet,69,69 | Ceramic products,USA | 69 | Ceramic products,6911,"6911 | Tableware, kitchenwa","USA | 6911 | Tableware, kitchenwa",69,6911,691190,Household articles & toilet articles (excl. ta...,Mining and Quarrying,2691,Manufacture of non-structural non-refractory c...,66612,Other household/toilet articles of porcelain/c...,34,NMM - Mineral products n.e.c.,37221,"Ceramic tableware, kitchenware, other househol...",4.3093,0.2321,both,0.168617,2835.333600,2059.8212,[Container],['Container'],9187.0,5708.535377,16185.602161


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 298
Total size of the fleet: 191 vessels
6 vessel types contained in the dataset: ['Refrigerated bulk' 'Container' 'General cargo' 'Bulk carrier'
 'Chemical tanker' 'Vehicle']
Total transport work capacity: 22337.8 million deadweight-tonne nautical miles
Total cargo transported: 6449153.9 tonnes
Total transport work: 12822.1 million tonne nautical miles
Average build year of the fleet: 2004
Total Energy Demands: 1554.2 EJ
Total CO2 produced: 119806.3 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,11899,7819759,616401000,LRMLW,430,LBR,LR,Liberia,Monrovia,GWOXB,624,GNB,GW,Guinea-Bissau,Bissau,2018-04-16 08:00:00,2018-04-20 18:00:00,106.0,187.0,293.0,608.880381,6.330766,0.000282,6.331047,29.7804,63.2808,93.0612,0.0,0.0,0.0,0.0,0.0,0.0,1.52612,2.702101,115.189967,202.879122,318.069089,002,Africa,202,Sub-Saharan Africa,low_income,1,0,002,Africa,202,Sub-Saharan Africa,low_income,1,1,4.228221,Refrigerated bulk,1753.0,dwt,2225.0,12.0,NaN,NaN,NaN,2,2000-5999,1979.0,Moldova,MSD,1,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,1.354759e+06,42743.106988,44267.474481,43546.40597,42913.993388,42293.297280,41684.317441,41063.130483,40477.927987,39881.207569,42777.347852,70689.103523,71008.514269,78629.560145,81287.323225,80332.766446,79164.835909,7888

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_dep_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_dep_voys_vess = int_dep_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_dep_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
                
                if int_dep_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_dep_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
    
        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the exporter.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "direct35_usd_dwtnm_"+year))

    if cost_int_match_desc_bool == False:
        trade_ex["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd
    trade_ex["direct_usd_"+year] = direct_usd
    
print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.direct_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_agg = trade_ex.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg["vol_kg_vtw_pct"] = round(100 * trade_ex_agg.vol_kg_vtw / trade_ex_agg.vol_kg_vtw.sum(), 1)
trade_ex_agg["fobvalue_pct"] = round(100 * trade_ex_agg.fobvalue / trade_ex_agg.fobvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_ex_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5)
))
trade_ex_agg_2 = trade_ex.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg_2["vol_kg_vtw_pct"] = round(100 * trade_ex_agg_2.vol_kg_vtw / trade_ex_agg_2.vol_kg_vtw.sum(), 1)
trade_ex_agg_2["fobvalue_pct"] = round(100 * trade_ex_agg_2.fobvalue / trade_ex_agg_2.fobvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_ex_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
trade_ex = trade_ex[(~trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_ex_na.primaryValue.sum(), round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Oil tanker']

Global voyages table used for the following vessel types found: 

['Other liquids tankers']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['7326 | Iron or steel; artic' '8311 | Wires, rods, tubes, '
 '8541 | Semiconductor device' '8703 | Motor cars and other'
 '8716 | Trailers and semi-tr' '4403 | Wood in the rough, w'
 '4911 | Printed matter, n.e.' '3926 | Articles of plastics'
 '4602 | Basketwork, wickerwo' '1905 | Bread, pastry, cakes'
 '2106 | Food preparations no' '2202 | Waters, including mi'
 '3004 | Medicaments; (not go' '9403 | Furniture and parts ']


Number of trade records present in dataset: 368
Number 

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,fobvalue_pct
0,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,56.4,7.5
1,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,16.4,21.0
2,"['Container', 'General cargo']",1. Primary Int. Inv. with Vess Type,16.4,36.1
3,['Container'],1. Primary Int. Inv. with Vess Type,8.3,21.3
4,"['Container', 'Chemical tanker']",1. Primary Int. Inv. with Vess Type,1.8,7.6


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,430,Liberia,694,Sierra Leone,000,World,Export,Sea,691110 | Tableware and kitche,691110,kg,5.0900,5.0900,NaN,1658.831,1658.831,Sierra Leone | 691110 | Tableware and kitche,69,69 | Ceramic products,Sierra Leone | 69 | Ceramic products,6911,"6911 | Tableware, kitchenwa","Sierra Leone | 6911 | Tableware, kitchenwa",69,6911,691110,"Tableware & kitchenware, of porcelain/china",Mining and Quarrying,2691,Manufacture of non-structural non-refractory c...,66611,Tableware & kitchenware of porcelain/china,34,NMM - Mineral products n.e.c.,37221,"Ceramic tableware, kitchenware, other househol...",4.3093,0.2321,both,0.003068,385.014675,5.0900,[Container],['Container'],677.0,420.668167,161.963418,1. Primary Int. Inv. with Vess Type,0.63952,0.638121,0.638266,0.638266,0.638911,0.638911,0.670637,1.211315,0.726113,0.688631,0.690786,0.705429,0.655914,1.176518,0.772135,0.712630,0.694773,0.728550,0.643002,1.305417,0.964412,0.845056,0.779379,0.760439,0.630327,1.380241,1.131854,1.044369,0.832190,0.816429,0.617889,1.416950,1.370688,1.393924,0.888256,0.858224,0.605204,1.386765,1.301817,1.410717,1.095285,0.874523,0.593255,1.217042,1.283715,1.339157,0.961620,0.884918,0.581072,1.118693,1.131315,1.309577,1.102825,0.891427
1,1,430,Liberia,842,USA,000,World,Export,Sea,691190 | Household and toilet,691190,kg,2059.8212,2059.8212,NaN,12216.000,12216.000,USA | 691190 | Household and toilet,69,69 | Ceramic products,USA | 69 | Ceramic products,6911,"6911 | Tableware, kitchenwa","USA | 6911 | Tableware, kitchenwa",69,6911,691190,Household articles & toilet articles (excl. ta...,Mining and Quarrying,2691,Manufacture of non-structural non-refractory c...,66612,Other household/toilet articles of porcelain/c...,34,NMM - Mineral products n.e.c.,37221,"Ceramic tableware, kitchenware, other househol...",4.3093,0.2321,both,0.168617,2835.333600,2059.8212,[Container],['Container'],9187.0,5708.535377,16185.602161,1. Primary Int. Inv. with Vess Type,63.90959,63.769792,63.784252,63.784252,63.848781,63.848781,67.019240,121.051163,72.563198,68.817462,69.032786,70.496112,65.547940,117.573815,77.162259,71.215742,69.431245,72.806687,64.257611,130.455165,96.377253,84.449580,77.886258,75.993484,62.990854,137.932588,113.110384,104.367657,83.163858,81.588769,61.747969,141.601024,136.977940,139.299976,88.766674,85.765462,60.480228,138.584528,130.095421,140.978132,109.455915,87.394267,59.286181,121.623517,128.286335,133.826865,96.098255,88.433138,58.068651,111.795136,113.056491,130.870865,110.209327,89.083595


## 3.4 Create Results Tables

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)

    
print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,3797599.60,3730526.26,3671999.62,3614535.30,3558231.23,3500813.38,3446724.69,3391608.73
1,Levy,6253583.82,6051704.64,6681290.45,7278401.81,7348657.55,7070603.37,6384325.05,5980831.98
2,'as-is',4032477.41,4226469.75,5085610.25,5891080.80,7003299.83,6780878.67,6700380.02,6083282.07
3,Brazil,3823064.71,3976419.62,4636615.29,5534765.60,7255024.51,7250365.05,7133033.94,6984324.51
4,LISCR,3834416.56,3906603.14,4350149.25,4572471.30,4940421.59,5830521.09,5519795.79,5879687.82
5,Direct,3949178.12,4050728.90,4192239.68,4444886.94,4630829.84,4700334.45,4744648.35,4779179.24



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.08,2.04,2.01,1.98,1.95,1.92,1.89,1.86
1,Levy,3.42,3.31,3.66,3.98,4.02,3.87,3.49,3.27
2,'as-is',2.21,2.31,2.78,3.22,3.83,3.71,3.67,3.33
3,Brazil,2.09,2.18,2.54,3.03,3.97,3.97,3.90,3.82
4,LISCR,2.10,2.14,2.38,2.50,2.70,3.19,3.02,3.22
5,Direct,2.16,2.22,2.29,2.43,2.54,2.57,2.60,2.62



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.072,0.071,0.070,0.069,0.068,0.067,0.066,0.065
1,Levy,0.119,0.115,0.127,0.139,0.140,0.135,0.122,0.114
2,'as-is',0.077,0.081,0.097,0.112,0.133,0.129,0.128,0.116
3,Brazil,0.073,0.076,0.088,0.105,0.138,0.138,0.136,0.133
4,LISCR,0.073,0.074,0.083,0.087,0.094,0.111,0.105,0.112
5,Direct,0.075,0.077,0.080,0.085,0.088,0.090,0.090,0.091


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:

    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)


print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,2455984.22,2321178.37,3009290.83,3663866.52,3790426.32,3569789.98,2937600.36,2589223.24
1,'as-is',234877.81,495943.49,1413610.64,2276545.50,3445068.60,3280065.29,3253655.33,2691673.34
2,Brazil,25465.11,245893.35,964615.67,1920230.31,3696793.28,3749551.67,3686309.25,3592715.78
3,LISCR,36816.96,176076.87,678149.63,957936.01,1382190.36,2329707.71,2073071.10,2488079.08
4,Direct,151578.52,320202.63,520240.06,830351.64,1072598.62,1199521.07,1297923.66,1387570.51



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,64.67,62.22,81.95,101.36,106.53,101.97,85.23,76.34
1,'as-is',6.18,13.29,38.50,62.98,96.82,93.69,94.40,79.36
2,Brazil,0.67,6.59,26.27,53.13,103.89,107.11,106.95,105.93
3,LISCR,0.97,4.72,18.47,26.50,38.84,66.55,60.15,73.36
4,Direct,3.99,8.58,14.17,22.97,30.14,34.26,37.66,40.91



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.34,1.27,1.65,2.01,2.07,1.95,1.61,1.42
1,'as-is',0.13,0.27,0.77,1.25,1.89,1.80,1.78,1.47
2,Brazil,0.01,0.13,0.53,1.05,2.02,2.05,2.02,1.97
3,LISCR,0.02,0.10,0.37,0.52,0.76,1.28,1.13,1.36
4,Direct,0.08,0.18,0.28,0.45,0.59,0.66,0.71,0.76



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.047,0.044,0.057,0.070,0.072,0.068,0.056,0.049
1,'as-is',0.004,0.009,0.027,0.043,0.066,0.062,0.062,0.051
2,Brazil,0.000,0.005,0.018,0.037,0.070,0.071,0.070,0.068
3,LISCR,0.001,0.003,0.013,0.018,0.026,0.044,0.039,0.047
4,Direct,0.003,0.006,0.010,0.016,0.020,0.023,0.025,0.026
